<a href="https://colab.research.google.com/github/tqd3pz/A01-data-wrangling/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [78]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

One row of my result is one line item inside an order, along with its order ID and vendor ID.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [79]:
import pandas as pd

lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

display(lines)

assert len(lines) == 4, f"Expected 4 rows, but got {len(lines)}"
assert lines['qty'].sum() == 7, f"Expected total quantity to be 7, but got {lines['qty'].sum()}"
print("Assertions passed!")

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


Assertions passed!


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [80]:
import pandas as pd

vendors = pd.DataFrame(vendors_json)
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)

display(joined)

unmatched = joined[joined['_merge'] == 'left_only']
for idx, row in unmatched.iterrows():
    print(f"Unmatched vendor ID: {row['vendor_id']} ({row['item']}) with {row['qty']} units attached.")

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


Unmatched vendor ID: V-07 (Hot Dog) with 3 units attached.


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [81]:
# Q3 — Total quantity per vendor name.
# The unmatched vendor V-07 has no name, so it appears as NaN.
# Using dropna=False keeps its quantity in our totals.

qty_per_vendor = joined.groupby('name', dropna=False)['qty'].sum().reset_index()
display(qty_per_vendor)

,name,qty
0,Cav Merch,1
1,Hoos Burgers,3
2,NaN,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [82]:
# Q4 — Which zone sold the most items?
# Calculating total quantity per zone, keeping dropna=False to account for the unmatched vendor's zone.
zone_sales = joined.groupby('zone', dropna=False)['qty'].sum().reset_index()
display(zone_sales)

# Explanation:
# Zone A sold 4 items, while the unmatched vendor accounts for 3.
# Since V-07 has no zone, those 3 items appear under NaN.
# Together, the total is 7 items.

,zone,qty
0,A,4
1,NaN,3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [86]:
import pandas as pd

ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},  # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

processed_ragged = []
for order in ragged_json:
    order_copy = order.copy()
    if 'lines' not in order_copy:
        order_copy['lines'] = []
    processed_ragged.append(order_copy)

ragged_flat = pd.json_normalize(
    processed_ragged,
    record_path='lines',
    meta=['order', 'vendor_id']
)

# Check whether each item has a quantity
ragged_flat['qty_provided'] = ragged_flat['qty'].notna()

# Order 5 does not have any line items, so it does not appear in the table.
# The missing quantity for Fries stays as NaN instead of 0 because
# we do not know how many were ordered.

display(ragged_flat)

,item,qty,order,vendor_id,qty_provided
0,Soda,2.0,4,V-01,True
1,Fries,NaN,6,V-01,False


### Q6 — Validate

In [84]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

Having one row per line item makes it easier to analyze individual items instead of looking at an entire order. For example, if we wanted to calculate how many sodas were sold, we could group the data by item and add the quantities together. If we had one row per order, we would have to go through the lists of items within each order first. Having one row per line item also makes it easier to join the data with vendor information and calculate totals.

b) In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

A missing quantity (NaN) means that the information was not provided or recorded, while a quantity of zero means that we know no units were ordered. Treating missing values as zero could lead to inaccurate results when analyzing the data. For example, when calculating the average quantity ordered, replacing missing values with zeros would lower the average even though we don't actually know those quantities. It could also create problems with inventory because a missing quantity should be checked rather than automatically treated as zero.
